In [25]:
import subprocess
import sys
import os
import pandas as pd
import numpy as np

# Enable HT
ORIGINAL_SPLIT = False 
BATCH_SIZE = 8
EPOCHS = 4

###
# HT
###
LEARNING_RATE = 2e-5
EPOCHS = 30
BATCH_SIZE = 24

###
# Cross Evaluation Phase
###

DATASET = 'rest-16'
                       
CV_SETTING = [
    [['GERestaurant', 'data_e2e', 'results_filtered', 'gbert-base', '500', 19], # 0.7160
    ['GERestaurant', 'data_e2e', 'results_filtered', 'gbert-base', '1000', 25], # 0.7495
    ['GERestaurant', 'data_e2e', 'results_filtered', 'gbert-base', 'full', 25], # 0.7727
    ['rest-16', 'data', 'results', 'uncased_L-12_H-768_A-12', '500', 8], # 0.6306
    ['rest-16', 'data', 'results', 'uncased_L-12_H-768_A-12', '1000', 22], # 0.6784
    ['rest-16', 'data', 'results', 'uncased_L-12_H-768_A-12', 'full', 27]], # 0.7345
    
    [['GERestaurant', 'data', 'results', 'gbert-base', '500', 24], # 0.6381
    ['GERestaurant', 'data', 'results', 'gbert-base', '1000', 13], # 0.6731
    ['GERestaurant', 'data', 'results', 'gbert-base', 'full', 30], # 0.7384
    ['rest-16', 'data_e2e', 'results_filtered', 'uncased_L-12_H-768_A-12', '500', 28], # 0.6860
    ['rest-16', 'data_e2e', 'results_filtered', 'uncased_L-12_H-768_A-12', '1000', 12], # 0.6920
    ['rest-16', 'data_e2e', 'results_filtered', 'uncased_L-12_H-768_A-12', 'full', 22]] # 0.7145
]

METHOD = 'tas_bert'
RESULTS_PATH = '../results'

col_names = ['dataset', 'lr_setting', 'split', 'learning_rate', 'epochs', 'f1-micro']
folder_names = [folder for folder in os.listdir(os.path.join(RESULTS_PATH, METHOD)) if os.path.isdir(os.path.join(RESULTS_PATH, METHOD, folder)) and folder != '.ipynb_checkpoints']

runs = []

for folder_name in folder_names:
    try:
        cond_parameters = folder_name.split('_')[:4]
        
        if cond_parameters[2] == '0':
            df = pd.read_csv(os.path.join(RESULTS_PATH, METHOD, folder_name, 'results.txt'), sep = '\t')
            df = df.set_index(df.columns[0])
    
            max_epoch = df['f1'].idxmax()
            
            cond_parameters.extend([max_epoch, df.loc[max_epoch, 'f1']])
            runs.append(cond_parameters)
    except:
        pass

results_all = pd.DataFrame(runs, columns = col_names)

# CV with Test Set
for DATA_PATH, OUTPUT_PATH in [['data', '../results/instructABSA']]:
    for LR_SETTING in ['full', '1000', '500']:

        results_sub = results_all[np.logical_and.reduce([results_all['lr_setting'] == LR_SETTING, results_all['dataset'] == DATASET])].sort_values(by = ['f1-micro'], ascending = False)
        results_sub = results_sub.reset_index()

        EPOCHS = int(results_sub.at[0, 'epochs'])



27
22
8


In [48]:
METHOD = 'hier_gcn'
RESULTS_PATH = '../results'

col_names = ['task', 'dataset', 'lr_setting', 'split', 'learning_rate', 'batch_size', 'epochs', 'f1-micro']
folder_names = [folder for folder in os.listdir(os.path.join(RESULTS_PATH, METHOD)) if os.path.isdir(os.path.join(RESULTS_PATH, METHOD, folder)) and folder != '.ipynb_checkpoints']

runs = []

for folder_name in folder_names:
    try:
        cond_parameters = folder_name.split('_')
        
        if cond_parameters[3] == '0':
            cond_params = cond_parameters.copy()
            
            with open(os.path.join(RESULTS_PATH, METHOD, folder_name, 'cate_eval_results.txt'), 'r') as f:
                f1 = f.readlines()[3].split(' = ')[1]
            
            cond_params.append(round(float(f1), 2))
            cond_params[0] = 'acd'
            runs.append(cond_params)

            cond_params = cond_parameters.copy()
            with open(os.path.join(RESULTS_PATH, METHOD, folder_name, 'eval_results.txt'), 'r') as f:
                f1 = f.readlines()[3].split(' = ')[1]
            
            cond_params.append(round(float(f1), 2))
            cond_params[0] = 'acsa'
            runs.append(cond_params)
    except:
        pass

results_all = pd.DataFrame(runs, columns = col_names)

In [50]:
results_all

,task,dataset,lr_setting,split,learning_rate,batch_size,epochs,f1-micro
0,acd,GERestaurant,1000,0,5e-05,8,20.0,0.89
1,acsa,GERestaurant,1000,0,5e-05,8,20.0,0.84
2,acd,GERestaurant,0,0,5e-05,8,20.0,0.93
3,acsa,GERestaurant,0,0,5e-05,8,20.0,0.88
4,acd,GERestaurant,500,0,5e-05,8,86.0,0.87
5,acsa,GERestaurant,500,0,5e-05,8,86.0,0.78
6,acd,rest-16,500,0,5e-05,8,68.0,0.72
7,acsa,rest-16,500,0,5e-05,8,68.0,0.61
8,acd,rest-16,0,0,5e-05,8,20.0,0.79
9,acsa,rest-16,0,0,5e-05,8,20.0,0.72


In [61]:
for TASK_NAME in ["GERestaurantACSA", "Rest16ACSA"]:
    for TASK in ['acd', 'acsa']:
        for LR_SETTING in [0, 1000, 500]:
            DATASET = 'GERestaurant' if TASK_NAME == 'GERestaurantACSA' else 'rest-16'
            results_sub = results_all[np.logical_and.reduce([results_all['task'] == TASK, results_all['lr_setting'] == str(LR_SETTING), results_all['dataset'] == DATASET, results_all['split'] == '0'])].sort_values(by = ['f1-micro'], ascending = False)
            results_sub = results_sub.reset_index()
            
            EPOCHS = int(eval(results_sub.at[0, 'epochs']))

            print(DATASET, TASK, LR_SETTING, EPOCHS, results_sub.at[0, 'f1-micro'])

GERestaurant acd 0 20 0.93
GERestaurant acd 1000 43 0.91
GERestaurant acd 500 86 0.87
GERestaurant acsa 0 20 0.88
GERestaurant acsa 1000 43 0.85
GERestaurant acsa 500 20 0.8
rest-16 acd 0 20 0.79
rest-16 acd 1000 20 0.79
rest-16 acd 500 20 0.73
rest-16 acsa 0 20 0.72
rest-16 acsa 1000 20 0.68
rest-16 acsa 500 20 0.65
